In [ ]:
import os
import gc
import sys

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import pyro
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from IPython.display import display

sns.set_context('paper')
rcParams.update({'font.family': 'Liberation Sans'})
rcParams.update({'font.size': 12})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, plot, utils, trajectory, metrics
import vgae, configs, dataset

%load_ext autoreload
%autoreload 2

#### Load data

In [ ]:
data_path = '../data/thymus/'
outdir = '../figures/'
sample_id = 'Mouse_Thymus1'

adata_rna = sc.read_h5ad(os.path.join(data_path, sample_id, 'adata_rna.h5'))
adata_protein = sc.read_h5ad(os.path.join(data_path, sample_id, 'adata_protein.h5'))
adata_protein.var_names_make_unique()

In [ ]:
sc.set_figure_params(scanpy=True, dpi_save=300, fontsize=10)

In [ ]:
# Ground-truth CMA annotations
t_cma = adata_rna.obs['CMA'].copy()
adata_rna.obs['CMA'] = (t_cma - t_cma.min()) / (t_cma.max() - t_cma.min())

fig, ax = plt.subplots(dpi=300)
ax = sq.pl.spatial_scatter(
    adata_rna, color='CML_Major',
    size=100, img=False,
    title=None, fig=fig, ax=ax, return_ax=True
)
ax.set_title('Ground-truth CMA lobule zones', fontsize=12)
# fig.savefig(os.path.join(outdir, 'LYNX_Fig3_CML.pdf'), bbox_inches='tight')

fig, ax = plt.subplots(dpi=300)
ax = sq.pl.spatial_scatter(
    adata_rna, color='CMA',
    cmap='RdBu_r', size=100, img=False, colorbar=False,
    title=None, fig=fig, ax=ax, return_ax=True
)
sm = ax.collections[0]
cbar = plt.colorbar(sm, ax=ax, shrink=0.9, aspect=20)
cbar.set_label(r'Pseudotime $(t)$', fontsize=8)
ax.set_title('Ground-truth CMA spatial gradient', fontsize=12)
# fig.savefig(os.path.join(outdir, 'LYNX_Fig3_CMA.pdf'), bbox_inches='tight')



#### PCA, Diffusion map, DPT

In [ ]:
# sc.pp.normalize_total(adata_rna)
# sc.pp.log1p(adata_rna)
# sc.pp.pca(adata_rna)
# sc.pp.neighbors(adata_rna)
# sc.tl.diffmap(adata_rna)

rep = 'X_pca'
curve = trajectory.get_curve(adata_rna, use_rep=rep, trim_radius_ratio=0.5)
trajectory.compute_pseudotime(adata_rna, curve, root_marker='Dcn', use_rep=rep)
adata_rna.obs['t_pca'] = adata_rna.obs['t'].copy()
adata_rna.obs.drop('t', inplace=True, axis=1)

rep = 'X_diffmap'
curve = trajectory.get_curve(adata_rna, use_rep=rep, trim_radius_ratio=0.5)
trajectory.compute_pseudotime(adata_rna, curve, root_marker='Dcn', use_rep=rep)
adata_rna.obs['t_diffmap'] = adata_rna.obs['t'].copy()
adata_rna.obs.drop('t', inplace=True, axis=1)

adata_rna.uns['iroot'] = adata_rna[:, 'Dcn'].X.argmax()
sc.tl.dpt(adata_rna)

t_dpt = adata_rna.obs['dpt_pseudotime'].copy()
t_dpt = (t_dpt-t_dpt.min()) / (t_dpt.max()-t_dpt.min())
adata_rna.obs['t_dpt'] = t_dpt
adata_rna.obs.drop('dpt_pseudotime', inplace=True, axis=1)

sq.pl.spatial_scatter(adata_rna, color=['t_pca', 't_diffmap', 't_dpt'], img=False, size=100, cmap='RdBu_r')

In [ ]:
t_cma = adata_rna.obs['CMA'].values

plot.disp_kde_scatter(
    t_cma, adata_rna.obs['t_pca'].values,
    size=.3, subset_ratio=0.5, logscale=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"PCA prediction $(t)$",
    title="Spatial gradient\n PCA vs. Ground-truth"
)

plot.disp_kde_scatter(
    t_cma, adata_rna.obs['t_diffmap'].values,
    size=.3, subset_ratio=0.5, logscale=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"Diffmap prediction $(t)$",
    title="Spatial gradient\n Diffmap vs. Ground-truth"
)

plot.disp_kde_scatter(
    t_cma, adata_rna.obs['t_dpt'].values,
    size=.3, subset_ratio=0.5, logscale=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"DPT prediction $(t)$",
    title="Spatial gradient\n DPT vs. Ground-truth"
)

#### LYNX

In [ ]:
n_latent = 6
adata_rna.obsm['X_z'] = np.load('../results/thymus/lynx_rna_{0}_{1}.npy'.format(n_latent, sample_id))
curve = trajectory.get_curve(adata_rna)
trajectory.compute_pseudotime(adata_rna, curve, root_marker='Dcn')

plot.disp_trajectory(
    adata_rna, cmap='RdBu',
    title='Principal Curve - LYNX'
)


adata_rna.obs['t_lynx'] = adata_rna.obs['t'].copy()
del adata_rna.obs['t']

In [ ]:
fig, ax = plt.subplots(dpi=300)
ax = sq.pl.spatial_scatter(
    adata_rna, color='t_lynx',
    cmap='RdBu_r', size=100, img=False, colorbar=False,
    title=None, fig=fig, ax=ax, return_ax=True
)
sm = ax.collections[0]
cbar = plt.colorbar(sm, ax=ax, shrink=0.9, aspect=20)
cbar.set_label(r'Pseudotime $(t)$', fontsize=8)
ax.set_title('Inferred spatial gradient - LYNX', fontsize=12)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_LYNX_spatial.pdf'), bbox_inches='tight')

In [ ]:
t_cma = adata_rna.obs['CMA'].values
t_lynx = adata_rna.obs['t_lynx'].values

fig, ax = plot.disp_kde_scatter(
    t_cma, t_lynx, subset_ratio=0.5,
    size=.3, logscale=False, show_plot=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"LYNX prediction $(t)$",
    title="Spatial gradient\n LYNX vs. Ground-truth"
)
ax.grid(False)
ax.plot([0, 1], [0, 1], ':', lw=0.75, color='k', alpha=0.8)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_LYNX_scatter.pdf'), bbox_inches='tight')

#### SIMVI

In [ ]:
adata_rna.obsm['X_simvi'] = np.load('../results/thymus/SIMVI_rna_s20.npy')
curve = trajectory.get_curve(adata_rna, use_rep='X_simvi', trim_radius_ratio=0.05)
trajectory.compute_pseudotime(adata_rna, curve, root_marker='Dcn', use_rep='X_simvi')

plot.disp_trajectory(
    adata_rna, cmap='RdBu', use_rep='X_simvi',
    title='Principal Curve - SIMVI'
)

adata_rna.obs['t_simvi'] = adata_rna.obs['t'].copy()
del adata_rna.obs['t']


In [ ]:
fig, ax = plt.subplots(dpi=300)
ax = sq.pl.spatial_scatter(
    adata_rna, color='t_simvi',
    cmap='RdBu_r', size=100, img=False, colorbar=False,
    title=None, fig=fig, ax=ax, return_ax=True
)
sm = ax.collections[0]
cbar = plt.colorbar(sm, ax=ax, shrink=0.9, aspect=20)
cbar.set_label(r'Pseudotime $(t)$', fontsize=8)
ax.set_title('Inferred spatial gradient - SIMVI', fontsize=12)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_SIMVI_spatial.pdf'), bbox_inches='tight')

In [ ]:
t_cma = adata_rna.obs['CMA'].values
t_simvi = adata_rna.obs['t_simvi'].values

fig, ax = plot.disp_kde_scatter(
    t_cma, t_simvi, subset_ratio=0.5,
    size=.3, logscale=False, show_plot=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"SIMVI prediction $(t)$",
    title="Spatial gradient\n SIMVI vs. Ground-truth"
)
ax.grid(False)
ax.plot([0, 1], [0, 1], ':', lw=0.75, color='k', alpha=0.8)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_SIMVI_scatter.pdf'), bbox_inches='tight')

#### scVI

In [ ]:
adata_rna.obsm['X_scvi'] = np.load('../results/thymus/scvi_10.npy')
curve = trajectory.get_curve(adata_rna, use_rep='X_scvi', trim_radius_ratio=1)
trajectory.compute_pseudotime(adata_rna, curve, root_marker='Dcn', use_rep='X_scvi')

plot.disp_trajectory(
    adata_rna, cmap='RdBu', use_rep='X_scvi',
    title='Principal Curve - X_scvi'
)

adata_rna.obs['t_scvi'] = adata_rna.obs['t'].copy()
del adata_rna.obs['t']

In [ ]:
fig, ax = plt.subplots(dpi=300)
ax = sq.pl.spatial_scatter(
    adata_rna, color='t_scvi',
    cmap='RdBu_r', size=100, img=False, colorbar=False,
    title=None, fig=fig, ax=ax, return_ax=True
)
sm = ax.collections[0]
cbar = plt.colorbar(sm, ax=ax, shrink=0.9, aspect=20)
cbar.set_label(r'Pseudotime $(t)$', fontsize=8)
ax.set_title('Inferred spatial gradient - scVI', fontsize=12)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_scVI_spatial.pdf'), bbox_inches='tight')

In [ ]:
t_cma = adata_rna.obs['CMA'].values
t_scvi = adata_rna.obs['t_scvi'].values

fig, ax = plot.disp_kde_scatter(
    t_cma, t_scvi, subset_ratio=0.5,
    size=.3, logscale=False, show_plot=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"scVI prediction $(t)$",
    title="Spatial gradient\n scVI vs. Ground-truth"
)
ax.grid(False)
ax.plot([0, 1], [0, 1], ':', lw=0.75, color='k', alpha=0.8)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_scVI_scatter.pdf'), bbox_inches='tight')

#### TotalVI

In [ ]:
adata_rna.obsm['X_totalvi'] = np.load('../results/thymus/totalvi_20.npy')
curve = trajectory.get_curve(adata_rna, use_rep='X_totalvi', trim_radius_ratio=1)
trajectory.compute_pseudotime(adata_rna, curve, root_marker='Dcn', use_rep='X_totalvi')

plot.disp_trajectory(
    adata_rna, cmap='RdBu', use_rep='X_totalvi',
    title='Principal Curve - LYNX'
)

adata_rna.obs['t_totalvi'] = adata_rna.obs['t'].copy()
del adata_rna.obs['t']


In [ ]:
fig, ax = plt.subplots(dpi=300)
ax = sq.pl.spatial_scatter(
    adata_rna, color='t_totalvi',
    cmap='RdBu_r', size=100, img=False, colorbar=False,
    title=None, fig=fig, ax=ax, return_ax=True
)
sm = ax.collections[0]
cbar = plt.colorbar(sm, ax=ax, shrink=0.9, aspect=20)
cbar.set_label(r'Pseudotime $(t)$', fontsize=8)
ax.set_title('Inferred spatial gradient - totalVI', fontsize=12)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_totalVI_spatial.pdf'), bbox_inches='tight')

In [ ]:
t_cma = adata_rna.obs['CMA'].values
t_totalvi = adata_rna.obs['t_totalvi'].values

fig, ax = plot.disp_kde_scatter(
    t_cma, t_totalvi, subset_ratio=0.5,
    size=.3, logscale=False, show_plot=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"totalVI prediction $(t)$",
    title="Spatial gradient\n totalVI vs. Ground-truth"
)
ax.grid(False)
ax.plot([0, 1], [0, 1], ':', lw=0.75, color='k', alpha=0.8)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_totalVI_scatter.pdf'), bbox_inches='tight')

#### GASTON

In [ ]:
adata_rna.obs['t_gaston'] = np.load('../results/thymus/GASTON_thymus_isodepth.npy')
adata_rna.obs['CML_GASTON'] = np.load('../results/thymus/GASTON_thymus_seg.npy')
adata_rna.obs['CML_GASTON'] = adata_rna.obs['CML_GASTON'].astype(np.int8).astype('category')

In [ ]:
fig, ax = plt.subplots(dpi=300)
vmin = np.quantile(adata_rna.obs['t_gaston'], 0.05)
vmax = np.quantile(adata_rna.obs['t_gaston'], 0.95)
ax = sq.pl.spatial_scatter(
    adata_rna, color='t_gaston',
    cmap='RdBu_r', size=100, img=False, colorbar=False,
    vmin=vmin, vmax=vmax,
    title=None, fig=fig, ax=ax, return_ax=True
)
sm = ax.collections[0]
cbar = plt.colorbar(sm, ax=ax, shrink=0.9, aspect=20)
cbar.set_label(r'Pseudotime $(t)$', fontsize=8)
ax.set_title('Inferred spatial gradient - GASTON', fontsize=12)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_GASTON_spatial.pdf'), bbox_inches='tight')

In [ ]:
t_cma = adata_rna.obs['CMA'].values
t_gaston = adata_rna.obs['t_gaston'].values

fig, ax = plot.disp_kde_scatter(
    t_cma, t_gaston, subset_ratio=0.5,
    size=.3, logscale=False, show_plot=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"GASTON prediction $(t)$",
    title="Spatial gradient\n GASTON vs. Ground-truth"
)
ax.grid(False)
ax.plot([0, 1], [0, 1], ':', lw=0.75, color='k', alpha=0.8)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_GASTON_scatter.pdf'), bbox_inches='tight')

#### SpaceFlow

In [ ]:
adata_rna.obs['t_spaceflow'] = pd.read_csv('../results/thymus/SpaceFlow_50_pseudotime.csv', index_col=0)['pSM_spaceflow']
adata_rna.obs['CML_SpaceFlow'] = pd.read_csv('../results/thymus/SpaceFlow_50_seg.csv', index_col=0)['gmm_cluster']
adata_rna.obs['CML_SpaceFlow'] = adata_rna.obs['CML_SpaceFlow'].astype(np.int8).astype('category')

In [ ]:
fig, ax = plt.subplots(dpi=300)
ax = sq.pl.spatial_scatter(
    adata_rna, color='t_spaceflow',
    cmap='RdBu_r', size=100, img=False, colorbar=False,
    title=None, fig=fig, ax=ax, return_ax=True
)
sm = ax.collections[0]
cbar = plt.colorbar(sm, ax=ax, shrink=0.9, aspect=20)
cbar.set_label(r'Pseudotime $(t)$', fontsize=8)
ax.set_title('Inferred spatial gradient - SpaceFlow', fontsize=12)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_SpaceFlow_spatial.pdf'), bbox_inches='tight')

In [ ]:
t_cma = adata_rna.obs['CMA'].values
t_spaceflow = adata_rna.obs['t_spaceflow'].values

fig, ax = plot.disp_kde_scatter(
    t_cma, t_spaceflow, subset_ratio=0.5,
    size=.3, logscale=False, show_plot=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"SpaceFlow prediction $(t)$",
    title="Spatial gradient\n SpaceFlow vs. Ground-truth"
)
ax.grid(False)
ax.plot([0, 1], [0, 1], ':', lw=0.75, color='k', alpha=0.8)
fig.savefig(os.path.join(outdir, 'LYNX_Fig3_SpaceFlow_scatter.pdf'), bbox_inches='tight')

#### Metrics Summary

(1). RMSE

In [ ]:
methods = ['DPT', 'scVI', 'TotalVI', 'GASTON', 'SpaceFlow','SIMVI', 'LYNX']
ts = ['t_dpt', 't_scvi', 't_totalvi', 't_spaceflow', 't_gaston', 't_simvi', 't_lynx']

custom_palette = [
    "#003f5c",  # Deep Blue
    "#ffa600",  # Bright Orange
    "#bc5090",  # Vivid Magenta
    "#ff6361",  # Bright Red
    "#9e3b1e",  # Coffee?
    "#28a745",  # Vivid Green
    "#d400ff",   # Bright Purple 
]

In [ ]:
from sklearn.metrics import mean_squared_error
from typing import List, Iterable

def compute_rmse(
    adata : sc.AnnData,
    use_rep : Iterable[str],
    y_true : Iterable[float],
    n_repeats : int = 1,
    ss_ratio : float = 0.01
):
    rmses = np.zeros((len(use_rep), n_repeats))
    n_obs = adata.shape[0]

    for j in range(n_repeats):
        # Random subset data points
        rand_indices = np.random.choice(np.arange(n_obs), int(ss_ratio*n_obs), replace=False)
        adata_ss = adata[rand_indices]

        for i, key in enumerate(use_rep):
            y_pred = adata_ss.obs[key].values
            rmses[i, j] = mean_squared_error(y_true[rand_indices], y_pred, squared=False)
        
        del adata_ss
        gc.collect()

    return rmses

In [ ]:
n_repeats = 100
rmses = compute_rmse(
    adata_rna, 
    y_true=t_cma,
    n_repeats=n_repeats,
    use_rep=['t_dpt', 't_scvi', 't_totalvi', 't_spaceflow', 't_gaston', 't_simvi', 't_lynx'],
)

plot_df = pd.DataFrame({
    'RMSE': rmses.flatten(),
    'Methods': np.repeat(methods, n_repeats),
})

plt.rcParams['axes.grid'] = False
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
sns.boxplot(plot_df, x='Methods', y='RMSE', palette=custom_palette, ax=ax)
ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Methods\n'+r"($lower\ is\ better$)", fontsize=16)
ax.set_ylabel('RMSE', fontsize=16)
ax.set_title("Root Mean Squared Error\n vs. ground-truth", fontsize=16)


from statannotations.Annotator import Annotator
pairs = [
    ('LYNX', 'SIMVI'), 
    ('LYNX', 'GASTON'),
    ('LYNX', 'SpaceFlow'),
    ('LYNX', 'TotalVI'),
    ('LYNX', 'scVI'),
    ('LYNX', 'DPT')
]
annotator = Annotator(ax, pairs, data=plot_df, x="Methods", y="RMSE")
annotator.configure(test='t-test_ind', text_format='star')
annotator.apply_and_annotate()

fig.savefig(os.path.join(outdir, 'LYNX_Fig3_RMSE_boxplot.pdf'), bbox_inches='tight')

del n_repeats
gc.collect()

(2). Moran's I (autocorrelation smoothness)

In [ ]:
# Moran's I
n_repeats = 100
morans = metrics.compute_moran_I(
    adata_rna, 
    n_repeats=n_repeats,
    ss_ratio=0.1,
    use_rep=['t_dpt', 't_scvi', 't_totalvi', 't_spaceflow', 't_gaston', 't_simvi', 't_lynx'],
)

custom_palette = [
    "#003f5c",  # Deep Blue
    "#ffa600",  # Bright Orange
    "#bc5090",  # Vivid Magenta
    "#ff6361",  # Bright Red
    "#9e3b1e",  # Coffee
    "#28a745",  # Vivid Green
    "#d400ff",  # Bright Purple 
]

plot_df = pd.DataFrame({
    'moran': morans.flatten(),
    'Methods': np.repeat(methods, morans.shape[-1])
})

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
sns.barplot(
    data=plot_df,
    x='Methods', y='moran', 
    errorbar='sd',
    capsize=.5,
    err_kws={'linewidth': .7},
    palette=custom_palette, ax=ax
)

ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Methods', fontsize=16)
ax.set_ylabel(r'I', fontsize=16)
ax.set_title("Moran's I: gradient smoothness", fontsize=16)

fig.savefig(os.path.join(outdir, 'Suppl3_moran_I_boxplot.pdf'), bbox_inches='tight')



(3). CMA lobule zonation clustering benchmark

In [ ]:
# Perform clustering for methods w/o default clustering results
for method in methods:
    if method in ['GASTON', 'SpaceFlow']:
        continue
    pseudotime_key = 't_'+method.lower()
    adata_rna.obs['t'] = adata_rna.obs[pseudotime_key].values.copy()
    utils.get_zonations(adata_rna, n_zones=4)
    adata_rna.obs['CML_'+method] = adata_rna.obs['zone'].astype(np.int8).astype('category')

del adata_rna.obs['t'], adata_rna.obs['zone']
del pseudotime_key
gc.collect()

In [ ]:
for method in methods:
    zone_key = 'CML_'+method
    sq.pl.spatial_scatter(
        adata_rna, color=zone_key,
        size=100, img=False,
        title='Inferred CMA lobule zones - {}'.format(method),
    )

del zone_key, method

In [ ]:
from sklearn.metrics import (
    adjusted_rand_score, 
    adjusted_mutual_info_score,
    normalized_mutual_info_score,
    homogeneity_completeness_v_measure
) 
                           
def evaluate_clustering_with_matching(y_true, y_pred):
    """
    Comprehensive clustering evaluation with optional Hungarian matching
    """
    results = {}
    results['ARI'] = adjusted_rand_score(y_true, y_pred)
    results['NMI'] = normalized_mutual_info_score(y_true, y_pred)
    results['AMI'] = adjusted_mutual_info_score(y_true, y_pred)
    h, c, v = homogeneity_completeness_v_measure(y_true, y_pred)
    results['Homogeneity'] = h
    results['Completeness'] = c
    results['V-measure'] = v

    return results

In [ ]:
cml_true = pd.Categorical(adata_rna.obs['CML_Major']).codes
for method in methods:
    cml_pred = adata_rna.obs['CML_'+method].values
    results = evaluate_clustering_with_matching(cml_true, cml_pred)
    print(method)
    print(results)

In [ ]:
# Collect clustering metrics for all methods
clustering_metrics = []

cml_true = pd.Categorical(adata_rna.obs['CML_Major']).codes

for method in methods:
    cml_pred = adata_rna.obs['CML_'+method].values
    results = evaluate_clustering_with_matching(cml_true, cml_pred)
    
    for metric_name, value in results.items():
        clustering_metrics.append({
            'Method': method,
            'Metric': metric_name,
            'Value': value
        })

# Create DataFrame
clustering_df = pd.DataFrame(clustering_metrics)

# Create summary plot
fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=200)
axes = axes.flatten()

metrics_to_plot = ['ARI', 'NMI', 'AMI', 'Homogeneity', 'Completeness', 'V-measure']

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    metric_data = clustering_df[clustering_df['Metric'] == metric]
    
    sns.barplot(
        data=metric_data,
        x='Method', y='Value',
        palette=custom_palette,
        ax=ax
    )
    
    ax.set_title(f'{metric}', fontsize=16)
    ax.set_xlabel('Methods\n'+r"($higher\ is\ better$)", fontsize=12)
    ax.set_ylabel(f'{metric} Score', fontsize=12)
    ax.spines[['right', 'top']].set_visible(False)
    
    # Rotate x-axis labels for better readability
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
fig.suptitle('CMA Lobule Zone Prediction vs Ground Truth', 
             fontsize=20, y=1.1)

fig.savefig(os.path.join(outdir, 'Suppl3_clustering_metrics_summary.pdf'), 
            bbox_inches='tight', dpi=200)

plt.show()

### Compile figure

In [ ]:
figure_outdir = os.path.abspath("../figures/Figure_3_thymus")

In [ ]:
%%bash -s $figure_outdir
cat<<'EOF' >$1.tex
\documentclass{article}
\usepackage[paperheight=270mm,paperwidth=210mm]{geometry}
\geometry{left=5mm,right=5mm,top=5mm,bottom=5mm}

\usepackage{silence}
\WarningsOff*

\usepackage[labelfont=bf]{caption}
\usepackage[rgb]{xcolor}

% Use traditional font packages so this compiles with tectonic/pdfLaTeX.
% (Removed fontspec + \setmainfont which require Xe/LuaTeX.)
\usepackage[utf8]{inputenc}
\usepackage[T1]{fontenc}
\usepackage{helvet}
\renewcommand{\familydefault}{\sfdefault}

\usepackage{graphicx}
\usepackage[export]{adjustbox}

% Allow underscores and more flexible graphics filenames
\usepackage{grffile}   % handles multiple dots/complex filenames
\usepackage{underscore}

\begin{document}

\noindent
\fontsize{11pt}{11pt}\selectfont

% ===== Panel A =====
\begin{minipage}[t]{0.73\textwidth}
\textbf{a}\\
\includegraphics[width=0.3\textwidth]{LYNX_Fig3_thymus_annot.pdf}
\includegraphics[width=0.32\textwidth]{LYNX_Fig3_CML.pdf}
\includegraphics[width=0.33\textwidth]{LYNX_Fig3_CMA.pdf}
\end{minipage}
% ===== Panel B =====
\begin{minipage}[t]{0.24\textwidth}
\textbf{b}\\
\includegraphics[width=\textwidth]{LYNX_Fig3_LYNX_spatial.pdf}
\end{minipage}

% ===== Panel C-F =====
\begin{minipage}[t]{0.24\textwidth}
\textbf{c}\\
\includegraphics[width=\textwidth]{LYNX_Fig3_SIMVI_spatial.pdf}
\end{minipage}
\begin{minipage}[t]{0.24\textwidth}
\textbf{d}\\
\includegraphics[width=\textwidth]{LYNX_Fig3_totalVI_spatial.pdf}
\end{minipage}
\begin{minipage}[t]{0.24\textwidth}
\textbf{e}\\
\includegraphics[width=\textwidth]{LYNX_Fig3_GASTON_spatial.pdf}
\end{minipage}
\begin{minipage}[t]{0.24\textwidth}
\textbf{f}\\
\includegraphics[width=\textwidth]{LYNX_Fig3_SpaceFlow_spatial.pdf}
\end{minipage}

% % ===== Panel D =====
% \begin{minipage}[t]{\textwidth}
% \textbf{g}\\
% \includegraphics[width=0.19\textwidth]{LYNX_Fig3_LYNX_scatter.pdf}
% \includegraphics[width=0.19\textwidth]{LYNX_Fig3_SIMVI_scatter.pdf}
% \includegraphics[width=0.19\textwidth]{LYNX_Fig3_totalVI_scatter.pdf}
% \includegraphics[width=0.19\textwidth]{LYNX_Fig3_GASTON_scatter.pdf}
% \includegraphics[width=0.19\textwidth]{LYNX_Fig3_SpaceFlow_scatter.pdf}
% \end{minipage}

% ===== Panel E-G =====
\begin{minipage}[t]{0.33\textwidth}
\textbf{g}\\
\includegraphics[width=\textwidth]{LYNX_Fig3_RMSE_boxplot.pdf}
\end{minipage}\hfill%
%
\begin{minipage}[t]{0.35\textwidth}
\textbf{h}\\
\includegraphics[width=\textwidth]{LYNX_Fig3_TEC_heatmap.pdf}
\end{minipage}\hfill%
%
\begin{minipage}[t]{0.25\textwidth}
\textbf{i}\\
\includegraphics[width=\textwidth]{LYNX_Fig3_CITE_dynamics1.pdf}
\includegraphics[width=\textwidth]{LYNX_Fig3_CITE_dynamics2.pdf}
\end{minipage}

\end{document}
EOF

/snap/bin/tectonic -c minimal $1.tex